# Hindi M-CDI-II (19–36 months): EDA and Wordbank-style norms

This notebook analyses included CDI-II participants only.

CDI-II measures **production only**. Comprehension was not collected and must remain missing, not zero.

All education analyses use **mother's education only**.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED = PROJECT_ROOT / "data" / "processed"
TABLES = PROJECT_ROOT / "outputs" / "tables"
FIGURES = PROJECT_ROOT / "outputs" / "figures"

TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)

## Load processed data

In [ ]:
participants = pd.read_csv(PROCESSED / "participant_analysis_cdi2.csv")
items = pd.read_csv(PROCESSED / "cdi2_items_long.csv")

participants = participants.query("included_final == 1").copy()
participants["age_month"] = np.floor(participants["age_months_exact"]).astype(int)
items = items.merge(
    participants[["participant_id", "age_month", "age_months_exact", "sex", "mother_education"]],
    on="participant_id",
    how="inner",
    validate="many_to_one"
)

print("Included CDI-II participants:", len(participants))
print("Item-level rows:", len(items))
participants.head()

In [ ]:
def quantile_table(df, score_col, age_col="age_month"):
    probs = [0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.75, 0.80, 0.90]
    grouped = df.groupby(age_col)[score_col]
    base = grouped.agg(n="count", mean="mean", sd="std", min="min", max="max").reset_index()
    q = grouped.quantile(probs).unstack().reset_index()
    q.columns = [age_col] + [f"p{int(p*100):02d}" for p in probs]
    return base.merge(q, on=age_col, how="left").sort_values(age_col)

def plot_sample_size(df, title, filename):
    counts = df.groupby("age_month").size()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(counts.index.astype(str), counts.values)
    ax.set(title=title, xlabel="Age (months)", ylabel="Included participants")
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=300, bbox_inches="tight")
    plt.show()

def plot_overall(df, score_col, title, filename):
    x = df["age_months_exact"].to_numpy()
    y = df[score_col].to_numpy()
    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]
    # Lightweight notebook preview. Replace with the project's validated GAM/spline model.
    degree = min(3, max(1, len(np.unique(x_sorted)) - 1))
    coef = np.polyfit(x_sorted, y_sorted, degree)
    grid = np.linspace(np.nanmin(x_sorted), np.nanmax(x_sorted), 200)
    fitted = np.polyval(coef, grid)
    fitted = np.clip(fitted, 0, np.nanmax(y))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(x, y, s=10, alpha=.25)
    ax.plot(grid, fitted, linewidth=3)
    ax.set(title=title, xlabel="Age (months)", ylabel="Vocabulary score")
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=300, bbox_inches="tight")
    plt.show()

def empirical_quantile_curves(df, score_col, probs, title, filename):
    tab = (
        df.groupby("age_month")[score_col]
          .quantile(probs)
          .unstack()
          .sort_index()
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    for p in probs:
        ax.plot(tab.index, tab[p], marker="o", label=f"{int(p*100)}")
    ax.set(title=title, xlabel="Age (months)", ylabel="Vocabulary score")
    ax.legend(title="Quantile", ncol=3)
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=300, bbox_inches="tight")
    plt.show()
    return tab.reset_index()

def grouped_median_trajectory(df, score_col, group_col, title, filename):
    tab = (
        df.dropna(subset=[group_col])
          .groupby(["age_month", group_col])[score_col]
          .median()
          .reset_index()
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    for label, part in tab.groupby(group_col):
        ax.plot(part["age_month"], part[score_col], marker="o", label=str(label))
    ax.set(title=title, xlabel="Age (months)", ylabel="Median vocabulary score")
    ax.legend(title=group_col.replace("_", " ").title())
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=300, bbox_inches="tight")
    plt.show()
    return tab

def item_age_proportions(items, response_col):
    return (
        items.groupby(["item_id", "word", "word_english", "age_month"])[response_col]
             .agg(n_observed="count", n_endorsed="sum", proportion="mean")
             .reset_index()
    )

def plot_selected_items(summary, selected_words, title, filename):
    plot_df = summary[summary["word_english"].isin(selected_words)].copy()
    fig, ax = plt.subplots(figsize=(10, 6))
    for label, part in plot_df.groupby("word_english"):
        ax.plot(part["age_month"], part["proportion"], marker="o", label=label)
    ax.set(title=title, xlabel="Age (months)", ylabel="Proportion of children")
    ax.set_ylim(0, 1)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=300, bbox_inches="tight")
    plt.show()

## Participant flow and exclusions

In [ ]:
flow = pd.read_csv(TABLES / "participant_flow.csv")
exclusions = pd.read_csv(TABLES / "exclusions.csv")
display(flow)
display(exclusions)

## Sample characteristics

In [ ]:
characteristics = pd.DataFrame({
    "Statistic": [
        "N", "Age mean", "Age SD", "Age median", "Age minimum", "Age maximum",
        "Production mean", "Production SD"
    ],
    "Value": [
        len(participants),
        participants["age_months_exact"].mean(),
        participants["age_months_exact"].std(),
        participants["age_months_exact"].median(),
        participants["age_months_exact"].min(),
        participants["age_months_exact"].max(),
        participants["cdi2_production_total"].mean(),
        participants["cdi2_production_total"].std(),
    ]
})
display(characteristics)
display(participants["sex"].value_counts(dropna=False).rename_axis("sex").reset_index(name="n"))
display(participants["mother_education"].value_counts(dropna=False).rename_axis("mother_education").reset_index(name="n"))
characteristics.to_csv(TABLES / "cdi2_sample_characteristics.csv", index=False)

## Sample size and score distribution by age

In [ ]:
plot_sample_size(participants, "CDI-II included sample size by age", "cdi2_sample_size_by_age.png")

fig, ax = plt.subplots(figsize=(11, 6))
participants.boxplot(column="cdi2_production_total", by="age_month", ax=ax)
ax.set_title("CDI-II production distribution by age")
ax.set_xlabel("Age (months)")
ax.set_ylabel("Productive vocabulary")
fig.suptitle("")
fig.tight_layout()
fig.savefig(FIGURES / "cdi2_production_distribution_by_age.png", dpi=300, bbox_inches="tight")
plt.show()

## Missingness and range checks

In [ ]:
required = [
    "participant_id", "age_months_exact", "sex", "mother_education",
    "hindi_percentage", "cdi2_production_total"
]
missingness = participants[required].isna().sum().rename("missing_n").reset_index(names="variable")
display(missingness)

if "comprehension_total" in participants.columns:
    assert participants["comprehension_total"].isna().all(), (
        "CDI-II comprehension must be missing because it was not measured."
    )

## Production norm table

In [ ]:
cdi2_prod_norms = quantile_table(participants, "cdi2_production_total")
display(cdi2_prod_norms)
cdi2_prod_norms.to_csv(TABLES / "cdi2_production_norms.csv", index=False)

## Overall production trajectory — no quantiles

In [ ]:
plot_overall(
    participants, "cdi2_production_total",
    "CDI-II production: overall trajectory",
    "cdi2_production_overall.png"
)

## Quartile curves: 25, 50, 75

In [ ]:
tab = empirical_quantile_curves(
    participants, "cdi2_production_total", [0.25,0.50,0.75],
    "CDI-II production: quartile curves",
    "cdi2_production_quartiles.png"
)
tab.to_csv(TABLES / "cdi2_production_quartiles.csv", index=False)

## Decile curves: 10–90

In [ ]:
tab = empirical_quantile_curves(
    participants, "cdi2_production_total",
    [0.10,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90],
    "CDI-II production: decile curves",
    "cdi2_production_deciles.png"
)
tab.to_csv(TABLES / "cdi2_production_deciles.csv", index=False)

## Selected quantiles: 10, 25, 50, 75, 90

In [ ]:
tab = empirical_quantile_curves(
    participants, "cdi2_production_total",
    [0.10,0.25,0.50,0.75,0.90],
    "CDI-II production: selected quantiles",
    "cdi2_production_selected_quantiles.png"
)
tab.to_csv(TABLES / "cdi2_production_selected_quantiles.csv", index=False)

## Sex-specific selected quantiles

In [ ]:
for sex, part in participants.dropna(subset=["sex"]).groupby("sex"):
    empirical_quantile_curves(
        part, "cdi2_production_total", [0.10,0.25,0.50,0.75,0.90],
        f"CDI-II production — {sex}",
        f"cdi2_production_selected_quantiles_{str(sex).lower()}.png"
    )

## Mother's education trajectories

In [ ]:
tab = grouped_median_trajectory(
    participants, "cdi2_production_total", "mother_education",
    "CDI-II production by mother's education",
    "cdi2_production_mother_education.png"
)
tab.to_csv(TABLES / "cdi2_production_mother_education.csv", index=False)

## Item production trajectories

In [ ]:
production_summary = item_age_proportions(items, "produce")
display(production_summary.head())
production_summary.to_csv(TABLES / "cdi2_item_production_by_age.csv", index=False)

selected_words = (
    production_summary.groupby("word_english")["n_observed"].sum()
    .sort_values(ascending=False).head(4).index.tolist()
)
plot_selected_items(
    production_summary, selected_words,
    "CDI-II selected item production trajectories",
    "cdi2_selected_item_production_trajectories.png"
)

## Category-level EDA

In [ ]:
category = (
    items.groupby(["participant_id", "age_month", "category_english"])
         .agg(production_proportion=("produce", "mean"))
         .reset_index()
)
category_summary = (
    category.groupby(["age_month", "category_english"])
            .agg(
                n=("participant_id", "nunique"),
                production_mean=("production_proportion", "mean")
            )
            .reset_index()
)
display(category_summary.head(20))
category_summary.to_csv(TABLES / "cdi2_category_trajectories.csv", index=False)

## Modelling note

The plotting helpers above provide transparent empirical previews. The production pipeline should replace polynomial preview smooths and empirical joined quantiles with the validated GAM/GAMLSS or smooth quantile-regression implementation described in the implementation specification.